# 🌰 Fındık — Otomatik Ön-Etiketleme (Grounding DINO + SAM2)

```
286646.zip (4.310 etiketsiz kare)
   ↓ Grounding DINO   metin promptu ile organları bul
   ↓ temizlik         NMS · iç içe kutu · alan denetimi
   ↓ SAM2             kutuyu maskeye oturt, SIKILAŞTIR
   ↓ CLIP             yalnızca 'Diseased' karelerde hastalık katmanı
   → labels_organ/    Leaf · Nut · Husk · Branch · Flower
   → labels_hastalik/ diseased_*
   ↓ İNSAN DÜZELTMESİ ← atlanamaz
   ↓ YOLO26 organ eğitimi
```

## Etiketleme kuralları (bu notebook bunlara göre ayarlandı)

- Her fındık kümesi **ayrı** kutu
- Her yaprak **ayrı** kutu
- Her dal (görünür ve anlamlıysa) **ayrı** kutu
- Hastalık varsa hastalıklı organ **ayrıca** etiketlenir

## ⚠️ Hastalık neden AYRI dosyada?

Aynı etiket dosyasına `leaf` **ve** `diseased_leaf` yazılırsa:

1. Aynı yaprak iki kutuya girer, NMS'te birbirini bastırır.
2. Hastalık kararı **organ modeline** yüklenir; uzman model gereksizleşir
   ve hiyerarşik mimari düz tek-modele çöker.

Çözüm: **aynı kutular, iki katman.**

| klasör | sınıflar | hangi model |
|---|---|---|
| `labels_organ/` | Leaf · Nut · Husk · Branch · Flower | organ modeli (YOLO26) |
| `labels_hastalik/` | diseased_* | uzman model |

Organ katmanı **sağlık bilmez**: sağlıklı yaprak da hastalıklı yaprak da
`Leaf` kutusudur. Bkz. `docs/MIMARI.md` § "Üçüncü fayda".

## ⚠️ Otomatik teşhis güvenilir DEĞİL — ölçüldü

CLIP sıfır-atış, ince taneli hastalık ayrımında taban çizgisine göre
yalnızca **+0.03** (yazı-tura). Bu yüzden hastalık katmanı **iki koşula
birden** bağlıdır:

1. Görüntü düzeyi etiketi `Diseased` olmalı (dosya adından)
2. Kırpıntının hastalık skoru eşiği geçmeli

`Healthy` karelerde hastalık kutusu **hiç** üretilmez. Hastalığın **adı**
verilmez — yalnızca "bu organda bozulma var". Ölçüm: `docs/EGITIM.md` § 2.6


In [ ]:
# 1️⃣ Ortam
!nvidia-smi -L
!pip install -q transformers

import torch, transformers
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
GPU = torch.cuda.is_available()
print('GPU         :', torch.cuda.get_device_name(0) if GPU else '⚠️ YOK — CPU')
if not GPU:
    print('   Runtime → Change runtime type → T4 GPU seçin.')
    print('   CPU\'da 4.310 kare ~7 saat sürer (ölçüldü: 5.5 sn/kare).')


In [ ]:
# 2️⃣ Depo ve veri
import os, subprocess
from pathlib import Path

REPO = Path('/content/SmartFarmStrawberryDisease')
if REPO.exists():
    os.chdir(REPO); subprocess.run(['git','pull','--rebase','origin','main'], check=False)
else:
    os.chdir('/content')
    subprocess.run(['git','clone',
                    'https://github.com/emrah1982/SmartFarmStrawberryDisease.git'],
                   check=True)
    os.chdir(REPO)
print('CWD:', Path.cwd())

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 286646.zip Drive'da nerede? Birkac yol denenir.
ADAYLAR = [
    Path('/content/drive/MyDrive/SmartFarmStrawberryDisease/datasets/findik/286646.zip'),
    Path('/content/drive/MyDrive/SmartFarmStrawberryDisease/286646.zip'),
    Path('/content/drive/MyDrive/datasets/findiks/286646.zip'),
    Path('/content/drive/MyDrive/286646.zip'),
]
ZIP = next((p for p in ADAYLAR if p.exists()), None)
if ZIP is None:
    print('⛔ 286646.zip bulunamadi. Drive\'da su yollardan birine koyun:')
    for p in ADAYLAR: print('   ', p)
else:
    print('✅ bulundu:', ZIP, '(%.0f MB)' % (ZIP.stat().st_size/1e6))


In [ ]:
# 3️⃣ Zip'i ac
import zipfile
from pathlib import Path

HAM = Path('datasets/findik/cotanak_ham')
HAM.mkdir(parents=True, exist_ok=True)
if not any(HAM.iterdir()):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(HAM)
kareler = sorted(p for p in HAM.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'})
print('%d kare' % len(kareler))

import collections, re
sag = collections.Counter(
    (re.search(r'(Diseased|Healthy)', p.name, re.I).group(1).lower()
     if re.search(r'(Diseased|Healthy)', p.name, re.I) else 'BILINMIYOR')
    for p in kareler)
print('goruntu duzeyi saglik etiketi:', dict(sag))
if sag.get('BILINMIYOR'):
    print('⚠️ Adinda Diseased/Healthy olmayan kare var → o karelerde')
    print('   hastalik katmani uretilmez (bilerek).')


In [ ]:
# 4️⃣ ÖNCE 60 KAREDE DENE — prompt ve eşikleri burada ayarla
# 4.310 kareyi çalıştırmadan önce çıktının doğru olduğunu GÖR.
# Prompt kötüyse 4.310 karelik çöp üretilir.

!python scripts/on_etiket_gdino.py datasets/findik/cotanak_ham \
    --urun findik --ad organ_gdino_deneme \
    --model IDEA-Research/grounding-dino-base \
    --kutu-esigi 0.30 --metin-esigi 0.25 \
    --hastalik --sinir 60


In [ ]:
# 5️⃣ Denemeyi GÖZLE kontrol et
# Rapordaki sayılar yetmez — kutular yerinde mi, BAK.
# Kalın çerçeve = hastalık katmanında da işaretli.

import random
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
import yaml

K = Path('datasets/findik/organ_gdino_deneme')
N = yaml.safe_load((K/'data.yaml').read_text(encoding='utf-8'))['names']
RENK = {'Leaf':'#00c853','Husk':'#e53935','Branch':'#1e88e5',
        'Nut':'#fb8c00','Flower':'#8e24aa'}

dolu = [p for p in sorted((K/'labels_organ').iterdir()) if p.read_text().strip()]
print('%d / %d karede kutu var' % (len(dolu), len(list((K/'labels_organ').iterdir()))))

fig, eks = plt.subplots(2, 3, figsize=(21, 14))
for ax, e in zip(eks.ravel(), random.Random(0).sample(dolu, min(6, len(dolu)))):
    img = next((K/'images').glob(e.stem + '.*'))
    with Image.open(img) as im:
        g, y = im.size; ax.imshow(im)
    h = K/'labels_hastalik'/(e.stem + '.txt')
    hasta = {tuple(s.split()[1:5]) for s in h.read_text().splitlines()
             if s.split()} if h.exists() else set()
    for s in e.read_text().splitlines():
        t = s.split()
        if len(t) != 5: continue
        ad = N[int(t[0])]; cx, cy, w, hh = map(float, t[1:])
        kalin = tuple(t[1:5]) in hasta
        ax.add_patch(patches.Rectangle(
            ((cx-w/2)*g, (cy-hh/2)*y), w*g, hh*y, fill=False,
            linewidth=4 if kalin else 2, edgecolor=RENK.get(ad, '#fff')))
        ax.text((cx-w/2)*g, (cy-hh/2)*y-5, ad + (' ⚠' if kalin else ''),
                color=RENK.get(ad, '#fff'), fontsize=10, weight='bold')
    ax.set_title(e.stem[:26], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

print(open(K/'KILAVUZ.md', encoding='utf-8').read()[:1500])


### Ayar rehberi — 4️⃣ hücresini yeniden çalıştırarak deneyin

| belirti | ne yapmalı |
|---|---|
| Çok az kutu, organlar atlanıyor | `--kutu-esigi 0.22` (düşür) |
| Çok fazla saçma kutu | `--kutu-esigi 0.38` (yükselt) |
| Kutular gevşek, nesneden büyük | `--sam` ekleyin (SAM2 sıkılaştırır) |
| Zuruf bulunmuyor | prompt'u `scripts/on_etiket_gdino.py: ORGAN_PROMPT`'ta değiştirin |
| Hastalık her şeyi işaretliyor | `--hastalik-esigi 0.70` (yükselt) |

> **Prompt yazarken:** küçük harf, cümleler **nokta** ile ayrılır.
> Uzun tamlamalar belirteç sınırında bölünür ve geri gelmeyen etiket
> üretir — ölçüldü: `'a cluster of hazelnuts in green husk.'` →
> `'a clusternut'`, `'##s husk'`. `'green hazelnut cluster.'` temiz çalıştı.


In [ ]:
# 6️⃣ TAM KOŞU — 4.310 kare
# T4'te ~15-25 dk. --sam eklerseniz 2-3 katına çıkar ama kutular sıkılaşır.
# Bu hücreyi ancak 5️⃣'teki görüntüler DÜZGÜNSE çalıştırın.

!python scripts/on_etiket_gdino.py datasets/findik/cotanak_ham \
    --urun findik --ad organ_gdino \
    --model IDEA-Research/grounding-dino-base \
    --kutu-esigi 0.30 --metin-esigi 0.25 \
    --sam --hastalik


In [ ]:
# 7️⃣ Sonucu denetle — eğitime uygun mu?
from pathlib import Path
import collections, importlib.util, sys

K = Path('datasets/findik/organ_gdino')
s = importlib.util.spec_from_file_location('io_', 'scripts/imgsz_oner.py')
io_ = importlib.util.module_from_spec(s); s.loader.exec_module(io_)

kutu_say = collections.Counter(); per_img = []; tam_kadraj = 0
benzersiz = collections.defaultdict(set)
import yaml
N = yaml.safe_load((K/'data.yaml').read_text(encoding='utf-8'))['names']
for e in (K/'labels_organ').iterdir():
    n = 0
    for satir in e.read_text().splitlines():
        c = io_.etiket_satiri(satir.split())
        if not c: continue
        n += 1; kutu_say[N[c[0]]] += 1
        benzersiz[N[c[0]]].add(tuple(round(v, 4) for v in c[1:]))
        if c[3] > 0.99 and c[4] > 0.99: tam_kadraj += 1
    per_img.append(n)

import numpy as np
a = np.array(per_img)
print('kutu/goruntu: medyan %d, ort %.2f, max %d, BOS %d (%%%.1f)'
      % (np.median(a), a.mean(), a.max(), (a == 0).sum(), 100*(a == 0).mean()))
print('tam-kadraj kutu: %d  %s' % (tam_kadraj, '⛔' if tam_kadraj else '✅'))
print()
print('%-10s %8s %10s  %s' % ('sinif', 'kutu', 'benzersiz', 'durum'))
for k, v in kutu_say.most_common():
    u = len(benzersiz[k])
    # Bir sinifin butun kutulari ayniysa o sinif BILGI TASIMIYOR demektir.
    print('%-10s %8d %10d  %s' % (k, v, u, '⛔ SABIT KUTU' if u < v*0.5 else '✅'))

# Alan tespiti: saha verisi mi?
!python scripts/imgsz_oner.py datasets/findik/organ_gdino


In [ ]:
# 8️⃣ Drive'a al — düzeltme dışarıda yapılacak
import shutil
from pathlib import Path

K = Path('datasets/findik/organ_gdino')
D = Path('/content/drive/MyDrive/SmartFarmStrawberryDisease/findik_aday_etiket')
D.parent.mkdir(parents=True, exist_ok=True)
if D.exists(): shutil.rmtree(D)
shutil.copytree(K, D)
print('kopyalandi:', D)
for alt in ('images', 'labels_organ', 'labels_hastalik'):
    if (D/alt).is_dir():
        print('  %-16s %d dosya' % (alt, len(list((D/alt).glob('*')))))
print('\nCVAT / Roboflow / Label Studio ile acip DUZELTIN.')


### 9️⃣ Düzeltirken

1. **Her küme, her yaprak, her dal AYRI kutu.** Model sık sık birkaç
   yaprağı tek kutuda topluyor — bölün.
2. **Eksikleri ekleyin.** Bulma oranı düşüktür; kenardaki ve arkadaki
   organlar çoğu zaman atlanıyor.
3. **Kutu görünen piksele oturur.** Arkada kalan kısmı tahmin edip
   büyütmeyin — model görmediğini öğrenemez.
4. **Kısa kenarı ~16 pikselden küçükse atlayın.** YOLO ızgarası 8 piksel
   adımlıdır; altındaki nesne zaten öğrenilemez.
5. **Arka plandaki bulanık organları silin.** Toprak ve gökyüzü
   üzerindeki kutular gürültüdür.
6. **Zuruf mu fındık mı?** Kabuk görünmüyorsa `Husk`; kabuk açığa
   çıkmışsa `Nut` yapın. Bu fotoğraflarda çoğunlukla `Husk` doğrudur.

### 🔟 Düzeltme sonrası — YOLO26 eğitimi

```bash
# 1. labels_organ/ -> duzeltilmis hali labels/ olarak
# 2. Sizinti ve bolme denetimi (ZORUNLU)
python scripts/harici_paket_duzelt.py datasets/findik/organ_gdino \
    --urun findik --ad organ_detection --kuru
# 3. Alan tespiti
python scripts/imgsz_oner.py datasets/findik/organ_detection
```

İkisi de temizse `StrawberryVision_Colab_Production.ipynb` içinde
`URUN = 'findik'`, `EGITILECEK = 'organ_detection'` yapıp **6️⃣** ile
eğitin. Model ailesi **YOLO26** (`configs/train_config.yaml`).

> Organ modeli kurulunca boru hattı fındık için açılır:
> `bahçe fotoğrafı → organ.pt → ROI kırp → uzman model → sonuç`
